In [1]:
import os

# Clear Kaggle TPU environment variables before importing torch_xla
os.environ.pop('TPU_PROCESS_ADDRESSES', None)
os.environ.pop('CLOUD_TPU_TASK_ID', None)

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import timm
import shutil

SCALES_TO_RUN = ["10%", "25%", "50%", "100%"]
EPOCHS        = 200     # Maximum cap; early stopping prevents overfitting on small scales
PATIENCE      = 20      # Number of epochs to wait for validation accuracy improvement
LR            = 5e-4
WEIGHT_DECAY  = 0.05

TEMPERATURE   = 4.0
W_TASK        = 0.6     # Weight for hard cross-entropy task loss
W_DISTILL     = 0.4     # Weight for soft KL divergence distillation loss
W_SEMANTIC    = 0.2     # Weight for feature alignment cosine distance loss

PROJ_DIM_IN   = 192
PROJ_DIM_OUT  = 2048    # Upgraded for ResNet-50 (2048-d avgpool)

In [2]:
DEVICE_TYPE = "GPU"   # "GPU" | "TPU"

class DeviceManager:
    """Unified stub so the training loop code handles both GPU and TPU seamlessly."""
    def __init__(self):
        self.type = DEVICE_TYPE
        if self.type == "TPU":
            import torch_xla
            import torch_xla.core.xla_model as xm
            import torch_xla.runtime as xr
            import torch_xla.distributed.parallel_loader as pl
            self.xm = xm
            self.xr = xr
            self.pl = pl
            self.device = torch_xla.device()
            self.world_size = xr.world_size()
            self.rank = xr.global_ordinal()
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.world_size = 1
            self.rank = 0

    def step(self, opt):
        if self.type == "TPU":
            self.xm.optimizer_step(opt)
            self.xm.mark_step()
        else:
            opt.step()

    def wrap_loader(self, loader):
        if self.type == "TPU":
            return self.pl.MpDeviceLoader(loader, self.device)
        return loader

    def is_master(self):
        return self.rank == 0

    def master_print(self, *args, **kwargs):
        if self.is_master():
            print(*args, **kwargs)

In [3]:
BACKUP_PATH = "/kaggle/input/datasets/totallyapoorv/horse2-res-crash-checkpoints/outputs" 
TARGET_DIR = "./outputs"

if os.path.exists(BACKUP_PATH):
    print("Restoring backup...")
    shutil.copytree(BACKUP_PATH, TARGET_DIR, dirs_exist_ok=True)
    print("Restore complete! Found files:")
    print("Checkpoints:", os.listdir(f"{TARGET_DIR}/checkpoints"))
    print("Results:", os.listdir(f"{TARGET_DIR}/results"))
else:
    print("Backup path not found. Check your dataset name.")

Restoring backup...
Restore complete! Found files:
Checkpoints: ['nb03_10pct.pth', 'nb03_100pct.pth', 'nb03_50pct.pth', 'nb03_25pct.pth']
Results: ['nb03_10pct.json', 'nb03_50pct.json', 'nb03_25pct.json']


In [4]:
TRAIN_DIR    = "/kaggle/input/datasets/melikechan/cifar100/cifar100/train"
TEST_DIR     = "/kaggle/input/datasets/melikechan/cifar100/cifar100/test"
TEACHER_CKPT = "/kaggle/input/datasets/totallyapoorv/resnet-teachermodel-50/teacher_resnet50.pth"

WORK_DIR = "./outputs"
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",     exist_ok=True)

In [5]:
SEED       = 67
BATCH_SIZE = 64
scale_map  = {"10%": 0.10, "25%": 0.25, "50%": 0.50, "100%": 1.0}
key_of     = lambda s: s.replace("%", "pct")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

def build_loaders(scale, ctx):
    tfm_tr = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    tfm_te = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    
    full = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=tfm_tr)
    test = torchvision.datasets.ImageFolder(TEST_DIR,  transform=tfm_te)
    
    idx = list(range(len(full)))
    random.Random(SEED).shuffle(idx)
    sub = torch.utils.data.Subset(full, idx[:int(len(full) * scale_map[scale])])
    
    sampler = torch.utils.data.distributed.DistributedSampler(
        sub, num_replicas=ctx.world_size, rank=ctx.rank, shuffle=True
    ) if ctx.world_size > 1 else None

    tr = torch.utils.data.DataLoader(
        sub, BATCH_SIZE, shuffle=(sampler is None),
        sampler=sampler, num_workers=2, pin_memory=False
    )
    te = torch.utils.data.DataLoader(
        test, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False
    )
    return tr, te, len(sub)

In [6]:
def evaluate(model, loader, ctx):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in ctx.wrap_loader(loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            out = model(x)
            
            # Handle standard tuple returns
            if isinstance(out, tuple): 
                # If tuple has 3 elements (NB04 WrappedStudent: cls, dist, attn)
                if len(out) == 3:
                    out = (out[0] + out[1]) / 2
                # If tuple has 2 elements (Standard Distilled output)
                else:
                    out = (out[0] + out[1]) / 2
            # Handle generic namespace returns
            elif hasattr(out, 'cls_logits') and hasattr(out, 'dist_logits'):
                out = (out.cls_logits + out.dist_logits) / 2
                
            correct += out.argmax(1).eq(y).sum().item()
            total   += y.size(0)
    
    if ctx.type == "TPU":
        correct = ctx.xm.mesh_reduce("test_correct", correct, sum)
        total = ctx.xm.mesh_reduce("test_total", total, sum)
        
    return 100. * correct / total

def save_ckpt(path, epoch, model, proj, opt, best_acc, history, scheduler=None, extras=None):
    d = dict(epoch=epoch, model=model.state_dict(), opt=opt.state_dict(),
             best_acc=best_acc, history=history)
    if proj is not None:      d["proj"]      = proj.state_dict()
    if scheduler is not None: d["scheduler"] = scheduler.state_dict()
    if extras:                d.update(extras)
    torch.save(d, path)

def load_ckpt(path, model, proj, opt, ctx, scheduler=None):
    d = torch.load(path, map_location='cpu')

    def strip_wrappers(state):
        cleaned = {}
        for k, v in state.items():
            new_key = k
            if new_key.startswith("ddp_module.module."): new_key = new_key[18:]
            if new_key.startswith("module."): new_key = new_key[7:]
            cleaned[new_key] = v
        return cleaned

    base_model = model.module if hasattr(model, 'module') else model
    base_model.load_state_dict(strip_wrappers(d["model"]))
    if proj is not None and "proj" in d:
        base_proj = proj.module if hasattr(proj, 'module') else proj
        base_proj.load_state_dict(strip_wrappers(d["proj"]))
    opt.load_state_dict(d["opt"])
    if scheduler is not None and "scheduler" in d:
        scheduler.load_state_dict(d["scheduler"])
    return d["epoch"] + 1, d["best_acc"], d.get("history", [])

In [7]:
'''def build_teacher(ctx):
    t = torchvision.models.resnet50(weights=None)
    t.fc = nn.Linear(t.fc.in_features, 100)
    t.load_state_dict(torch.load(TEACHER_CKPT, map_location='cpu'))
    t = WrappedTeacher(t).to(ctx.device).eval()
    for p in t.parameters():
        p.requires_grad_(False)
    # NO DataParallel — teacher runs in no_grad every step.
    # DataParallel broadcasts 102MB of ResNet-50 params to GPU1 each call,
    # causing caching-allocator fragmentation that OOMs at large scales.
    return t
def build_student_and_proj(ctx):
    m = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=False, num_classes=100)
    m.set_distilled_training(True)
    m = m.to(ctx.device)
    
    proj = nn.Linear(PROJ_DIM_IN, PROJ_DIM_OUT).to(ctx.device)
    cache = {}

    def t_hook(module, inp, out):
        cache["t_pool"] = out.view(out.size(0), -1)

    def s_hook(module, inp, out):
        cache["s_dist"] = inp[0]

    if ctx.type == "GPU" and ctx.world_size > 1:
        m = DDPWrapper(m, device_ids=[ctx.rank])
        proj = DDPWrapper(proj, device_ids=[ctx.rank])

    if ctx.is_master():
        total_params = sum(p.numel() for p in m.parameters()) + sum(p.numel() for p in proj.parameters())
        print(f"  Student + Proj params: {total_params/1e6:.2f}M")
        
    return m, proj, cache, t_hook, s_hook'''

'def build_teacher(ctx):\n    t = torchvision.models.resnet50(weights=None)\n    t.fc = nn.Linear(t.fc.in_features, 100)\n    t.load_state_dict(torch.load(TEACHER_CKPT, map_location=\'cpu\'))\n    t = WrappedTeacher(t).to(ctx.device).eval()\n    for p in t.parameters():\n        p.requires_grad_(False)\n    # NO DataParallel — teacher runs in no_grad every step.\n    # DataParallel broadcasts 102MB of ResNet-50 params to GPU1 each call,\n    # causing caching-allocator fragmentation that OOMs at large scales.\n    return t\ndef build_student_and_proj(ctx):\n    m = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=False, num_classes=100)\n    m.set_distilled_training(True)\n    m = m.to(ctx.device)\n    \n    proj = nn.Linear(PROJ_DIM_IN, PROJ_DIM_OUT).to(ctx.device)\n    cache = {}\n\n    def t_hook(module, inp, out):\n        cache["t_pool"] = out.view(out.size(0), -1)\n\n    def s_hook(module, inp, out):\n        cache["s_dist"] = inp[0]\n\n    if ctx.type == "GPU" and

In [8]:
def kd_loss(cls_l, dist_l, t_logits, y, T=TEMPERATURE):
    task    = F.cross_entropy(cls_l, y)
    soft_t  = F.softmax(t_logits / T, dim=1)
    log_s   = F.log_softmax(dist_l  / T, dim=1)
    distill = F.kl_div(log_s, soft_t, reduction="batchmean") * (T**2)
    return task, distill

def semantic_loss(t_pool, s_dist, proj):
    s_f = proj(s_dist)
    cos = F.cosine_similarity(t_pool, s_f, dim=1)
    return 1.0 - cos.mean()

In [9]:
DEVICE_TYPE = "GPU"   # "GPU" | "TPU"

class DeviceManager:
    """Unified stub so the training loop code handles both multi-GPU DP and TPU seamlessly."""
    def __init__(self):
        self.type = DEVICE_TYPE
        if self.type == "TPU":
            import torch_xla
            import torch_xla.core.xla_model as xm
            import torch_xla.runtime as xr
            import torch_xla.distributed.parallel_loader as pl
            self.xm = xm
            self.xr = xr
            self.pl = pl
            self.device = torch_xla.device()
            self.world_size = xr.world_size()
            self.rank = xr.global_ordinal()
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.world_size = 1
            self.rank = 0

    def step(self, opt):
        if self.type == "TPU":
            self.xm.optimizer_step(opt)
            self.xm.mark_step()
        else:
            opt.step()

    def wrap_loader(self, loader):
        if self.type == "TPU":
            return self.pl.MpDeviceLoader(loader, self.device)
        return loader

    def is_master(self):
        return self.rank == 0

    def master_print(self, *args, **kwargs):
        if self.is_master():
            print(*args, **kwargs)

In [10]:
def train_one_scale(index, scale):
    ctx = DeviceManager()
    key  = key_of(scale)
    ckpt = f"{WORK_DIR}/checkpoints/nb03_{key}.pth"
    rp   = f"{WORK_DIR}/results/nb03_{key}.json"

    if os.path.exists(rp) and json.load(open(rp)).get("completed"):
        r = json.load(open(rp))
        ctx.master_print(f"[{scale}] already done — best {r['best_acc']:.2f}%")
        return r

    tr_loader, te_loader, n_train = build_loaders(scale, ctx)
    
    teacher = build_teacher(ctx)
    student, proj = build_student_and_proj(ctx)

    params = list(student.parameters()) + list(proj.parameters())
    opt     = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    start_epoch, best_acc, history = 0, 0.0, []
    epochs_no_improve = 0

    if os.path.exists(ckpt):
        d = torch.load(ckpt, map_location='cpu')
        start_epoch, best_acc, history = load_ckpt(ckpt, student, proj, opt, ctx)
        
        # Restore scheduler to correct LR position
        if "scheduler" in d:
            scheduler.load_state_dict(d["scheduler"])
        
        # Restore patience counter — don't reset to 0 on resume
        epochs_no_improve = d.get("epochs_no_improve", 0)
        
        ctx.master_print(
            f"[{scale}] resumed at epoch {start_epoch+1} | "
            f"best={best_acc:.2f}% | wait={epochs_no_improve}/{PATIENCE}"
        )

    ctx.master_print(f"[{scale}] training on {n_train} images")
    for epoch in range(start_epoch, EPOCHS):
        if ctx.type == "TPU" and hasattr(tr_loader, 'sampler') and hasattr(tr_loader.sampler, 'set_epoch'):
            tr_loader.sampler.set_epoch(epoch)
            
        student.train()
        proj.train()
        t0 = time.time()
        
        # Standard Python float tracking is safest for CPU memory on GPUs
        sums = {"total": 0., "task": 0., "distill": 0., "semantic": 0.}

        for x, y in ctx.wrap_loader(tr_loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            opt.zero_grad()
            
            with torch.no_grad():
                t_logits, t_pool = teacher(x) 
                
            cls_l, dist_l, s_dist = student(x) 
                
            tl, dl = kd_loss(cls_l, dist_l, t_logits, y)
            sl     = semantic_loss(t_pool, s_dist, proj)
            loss   = (W_TASK * tl) + (W_DISTILL * dl) + (W_SEMANTIC * sl)
            
            loss.backward()
            ctx.step(opt)
            
            # Immediately extract floats to prevent graph accumulation
            sums["total"] += loss.item()
            sums["task"] += tl.item()
            sums["distill"] += dl.item()
            sums["semantic"] += sl.item()
            
            # FORCE python garbage collection of heavy tensors to prevent RAM leaks
            del x, y, t_logits, t_pool, cls_l, dist_l, s_dist, tl, dl, sl, loss

        epoch_time = time.time() - t0
        scheduler.step()
        
        nb_ = len(tr_loader)
        
        if ctx.type == "GPU" and ctx.world_size > 1:
            import torch.distributed as dist
            metrics = torch.tensor([sums["total"], sums["task"], sums["distill"], sums["semantic"]], dtype=torch.float32, device=ctx.device)
            dist.all_reduce(metrics, op=dist.ReduceOp.SUM)
            avg_total, avg_task, avg_dist, avg_sem = (metrics / (nb_ * ctx.world_size)).tolist()
        else:
            avg_total = sums["total"] / nb_
            avg_task  = sums["task"] / nb_
            avg_dist  = sums["distill"] / nb_
            avg_sem   = sums["semantic"] / nb_
            
        avg_cos = 1.0 - avg_sem
        val_acc = evaluate(student, te_loader, ctx)
        
        if val_acc > best_acc:
            best_acc = val_acc
            epochs_no_improve = 0
            is_best = True
        else:
            epochs_no_improve += 1
            is_best = False

        # ...inside the epoch loop, after evaluate and before early stopping check...

        if ctx.is_master():
            rec = {
                "epoch": epoch, "total": avg_total, "task": avg_task,
                "distill": avg_dist, "semantic": avg_sem, "cos_sim": avg_cos,
                "val_acc": val_acc, "epoch_time": epoch_time
            }
            history.append(rec)
            ctx.master_print(
                f"[{scale}] E{epoch+1}/{EPOCHS} | total={avg_total:.4f} "
                f"task={avg_task:.4f} kl={avg_dist:.4f} sem={avg_sem:.4f} "
                f"cos={avg_cos:.4f} | val={val_acc:.2f}% | "
                f"{epoch_time:.0f}s | wait={epochs_no_improve}/{PATIENCE}"
            )
            if is_best:
                # Single save with scheduler state included
                save_ckpt(ckpt, epoch, student, proj, opt, best_acc, history,
                          extras={"scheduler": scheduler.state_dict(),
                                  "epochs_no_improve": epochs_no_improve})
    

        import gc
        torch.cuda.empty_cache()
        gc.collect()
        # ─────────────────────────────────────────────────────────────────────────
    
        if epochs_no_improve >= PATIENCE:
            ctx.master_print(f"[{scale}] Early stopping at epoch {epoch+1}!")
            break

    if ctx.is_master():
        result = {"scale": scale, "n_train": n_train, "final_acc": history[-1]["val_acc"] if history else best_acc,
                  "best_acc": best_acc, "history": history, "completed": True}
        json.dump(result, open(rp, "w"), indent=2)
        ctx.master_print(f"[{scale}] DONE — final {result['final_acc']:.2f}% | best {best_acc:.2f}%")
        return result
    return None

In [11]:
class WrappedTeacher(nn.Module):
    """Explicit wrapper to return both logits and intermediate features safely across devices."""
    def __init__(self, base_teacher):
        super().__init__()
        self.base_teacher = base_teacher
        
    def forward(self, x):
        x = self.base_teacher.conv1(x)
        x = self.base_teacher.bn1(x)
        x = self.base_teacher.relu(x)
        x = self.base_teacher.maxpool(x)
        x = self.base_teacher.layer1(x)
        x = self.base_teacher.layer2(x)
        x = self.base_teacher.layer3(x)
        x = self.base_teacher.layer4(x)
        
        pool_feat = self.base_teacher.avgpool(x)
        pool_feat = torch.flatten(pool_feat, 1)
        logits = self.base_teacher.fc(pool_feat)
        return logits, pool_feat

class WrappedStudent(nn.Module):
    """Explicit wrapper to isolate and pass distillation token states up to DataParallel engines."""
    def __init__(self, base_student):
        super().__init__()
        self.base_student = base_student
        
    def forward(self, x):
        features = self.base_student.forward_features(x)
        cls_token = features[:, 0]
        dist_token = features[:, 1]
        
        cls_logits = self.base_student.head(cls_token)
        dist_logits = self.base_student.head_dist(dist_token)
        return cls_logits, dist_logits, dist_token

def build_teacher(ctx):
    t = torchvision.models.resnet50(weights=None)
    t.fc = nn.Linear(t.fc.in_features, 100)
    t.load_state_dict(torch.load(TEACHER_CKPT, map_location='cpu'))
    t = WrappedTeacher(t).to(ctx.device).eval()
    for p in t.parameters():
        p.requires_grad_(False)
    return t

def build_student_and_proj(ctx):
    m = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=False, num_classes=100)
    m = WrappedStudent(m).to(ctx.device)
    
    proj = nn.Linear(PROJ_DIM_IN, PROJ_DIM_OUT).to(ctx.device)
    
    if ctx.type == "GPU" and torch.cuda.device_count() > 1:
        m = nn.DataParallel(m)
        proj = nn.DataParallel(proj)

    if ctx.is_master():
        total_params = sum(p.numel() for p in m.parameters()) + sum(p.numel() for p in proj.parameters())
        print(f"  Student + Proj params: {total_params/1e6:.2f}M")
        
    return m, proj

In [12]:
def _mp_fn(index, scales):
    for scale in scales:
        train_one_scale(index, scale)

if __name__ == "__main__":
    if DEVICE_TYPE == "TPU":
        import torch_xla.distributed.xla_multiprocessing as xmp
        xmp.spawn(_mp_fn, args=(SCALES_TO_RUN,), start_method='fork')
    else:
        # Avoids multiprocessing for GPUs entirely.
        # DataParallel handles multi-threading inside this single process smoothly.
        _mp_fn(0, SCALES_TO_RUN)
    print("All scales complete.")

[10%] already done — best 18.70%
[25%] already done — best 28.60%
[50%] already done — best 35.69%
  Student + Proj params: 5.96M
[100%] resumed at epoch 52 | best=46.39% | wait=0/20
[100%] training on 50000 images


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return F.linear(input, self.weight, self.bias)


[100%] E52/200 | total=0.7258 task=0.0204 kl=1.6889 sem=0.1897 cos=0.8103 | val=45.77% | 253s | wait=1/20
[100%] E53/200 | total=1.7125 task=0.2249 kl=3.8352 sem=0.2176 cos=0.7824 | val=43.58% | 254s | wait=2/20
[100%] E54/200 | total=0.9658 task=0.0565 kl=2.2305 sem=0.1986 cos=0.8014 | val=45.75% | 254s | wait=3/20
[100%] E55/200 | total=0.7222 task=0.0210 kl=1.6801 sem=0.1878 cos=0.8122 | val=46.43% | 254s | wait=0/20
[100%] E56/200 | total=1.5669 task=0.1976 kl=3.5149 sem=0.2116 cos=0.7884 | val=44.74% | 254s | wait=1/20
[100%] E57/200 | total=1.0007 task=0.0654 kl=2.3046 sem=0.1982 cos=0.8018 | val=45.12% | 254s | wait=2/20
[100%] E58/200 | total=0.7868 task=0.0321 kl=1.8243 sem=0.1894 cos=0.8106 | val=44.71% | 254s | wait=3/20
[100%] E59/200 | total=1.4225 task=0.1631 kl=3.2074 sem=0.2084 cos=0.7916 | val=43.12% | 254s | wait=4/20
[100%] E60/200 | total=0.9688 task=0.0622 kl=2.2310 sem=0.1956 cos=0.8044 | val=45.24% | 254s | wait=5/20
[100%] E61/200 | total=0.8228 task=0.0405 kl=1

In [13]:
import os
import json

SCALES_TO_RUN = ["10%", "25%", "50%", "100%"]
WORK_DIR      = "./outputs"
key_of        = lambda s: s.replace("%", "pct")

print(f"\n{'Scale':<8} {'Final Acc':>10} {'Best Acc':>10} {'Train (s)':>12}")
for scale in SCALES_TO_RUN:
    rp = f"{WORK_DIR}/results/nb03_{key_of(scale)}.json"
    if os.path.exists(rp):
        try:
            with open(rp, "r") as f:
                r = json.load(f)
            wall = sum(h.get("epoch_time", 0) for h in r.get("history", []))
            print(f"{scale:<8} {r.get('final_acc', 0.0):>9.2f}% {r.get('best_acc', 0.0):>9.2f}% {wall:>11.0f}s")
        except Exception:
            print(f"{scale:<8} error reading results json")
    else:
        print(f"{scale:<8} not yet complete")


Scale     Final Acc   Best Acc    Train (s)
10%          18.03%     18.70%         962s
25%          27.66%     28.60%        3964s
50%          34.94%     35.69%        3761s
100%         49.68%     49.83%       33831s
